# Gaussian discriminant analysis

_Machine Learning_

**Model each class as a bell curve, then flip it with Bayes to classify.**

There are two styles of classifier.

     Discriminative models learn the boundary directly (like logistic regression).

---

This notebook builds Gaussian discriminant analysis one step at a time. Run each cell, read the note above it, and watch how modeling each class as a bell curve turns into a classifier. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. Each row is one example; the columns are its features, plus a class label.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)
print("rows x columns:", data.frame.shape)
print("feature columns:", list(data.data.columns))
print("target classes :", list(data.target_names))
data.frame.head()

## Reference implementation — scikit-learn

GDA models each class as a Gaussian (a bell curve) and uses Bayes' rule to pick the most likely class. We build it in three steps: (1) make a labelled dataset and split it, (2) fit the two GDA variants, (3) read off how well each one classifies.

### Step 1 — Make a dataset and split it

We generate 400 examples with 5 features, 3 of which actually carry signal. Then we hold out 25% of the data as a **test set** — fitting on the training rows and scoring on the unseen test rows keeps the accuracy honest rather than memorised.

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# 400 examples, 5 features (3 informative, 0 redundant), reproducible via random_state.
X, y = make_classification(
    n_samples=400,
    n_features=5,
    n_informative=3,
    n_redundant=0,
    random_state=0,
)

# Hold out 25% of the rows to test on data the model never saw during fitting.
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

print("train rows:", Xtr.shape[0], " test rows:", Xte.shape[0])

### Step 2 — Fit LDA and QDA

Both variants model each class as a Gaussian; they differ in the **covariance** (the shape of the bell curve).

- **LDA** forces all classes to *share one* covariance, giving a straight (linear) boundary.
- **QDA** lets each class have its *own* covariance, giving a curved (quadratic) boundary.

In [ ]:
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)

lda = LinearDiscriminantAnalysis().fit(Xtr, ytr)     # shared covariance
qda = QuadraticDiscriminantAnalysis().fit(Xtr, ytr)  # per-class covariance

### Step 3 — Score and inspect the priors

We score each fitted model on the held-out test set to compare their accuracy. We also print the **class priors** LDA learned — the fraction of training examples in each class, which is the `P(class)` term Bayes' rule multiplies by the Gaussian likelihood.

In [ ]:
print("LDA test accuracy:", round(lda.score(Xte, yte), 3))
print("QDA test accuracy:", round(qda.score(Xte, yte), 3))
print("class priors:", np.round(lda.priors_, 3))

## Visualize the data & results

_Modeling each tumor class as a Gaussian: does shared (LDA) or per-class (QDA) covariance classify breast-cancer better?_

### Step 1 — Load and standardise the breast-cancer data

We switch to the real breast-cancer scans. We **standardise** every feature (subtract the mean, divide by the std) so no single large-valued column dominates the Gaussian fit, then split into train and test sets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)   # zero mean, unit variance per feature
y = bc.target

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0)

### Step 2 — Plot the two tumor classes in 2D

The data has 30 features, too many to see at once. We use **PCA** to project it down to its two most-spread directions, then scatter the malignant and benign points so we can eyeball how separable they are.

In [ ]:
from sklearn.decomposition import PCA

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

# Project the 30-D data down to 2 dimensions for plotting.
P = PCA(n_components=2, random_state=0).fit_transform(X)

for label, color, name in [(0, "#ff7b72", "malignant"), (1, "#7ee787", "benign")]:
    pts = P[y == label]
    ax1.scatter(pts[:, 0], pts[:, 1], c=color, label=name, edgecolor="k")

ax1.set_xlabel("PCA component 1")
ax1.set_ylabel("PCA component 2")
ax1.set_title("Two tumor classes")
ax1.legend()

### Step 3 — Compare LDA vs QDA accuracy

Finally we fit both variants on the standardised training data and score them on the test set, drawing the two accuracies as bars. This shows directly whether the shared-covariance (LDA) or per-class (QDA) assumption fits these tumours better.

In [ ]:
from sklearn.discriminant_analysis import (
    LinearDiscriminantAnalysis,
    QuadraticDiscriminantAnalysis,
)

lda = LinearDiscriminantAnalysis().fit(Xtr, ytr).score(Xte, yte)
qda = QuadraticDiscriminantAnalysis().fit(Xtr, ytr).score(Xte, yte)

ax2.bar(["LDA", "QDA"], [lda, qda], color=["#4ea1ff", "#c89bff"])
ax2.set_title("Test accuracy: LDA vs QDA")
plt.show()